In [1]:
!pip install -r requirements_imdb_faiss_autogen_colab_gptoss20b.txt

In [3]:
!pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [4]:
!pip install -U torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.13.0+cu130
True
NVIDIA A100-SXM4-80GB


In [4]:
import argparse
import asyncio
import hashlib
import json
import os
import re
import sys
from pathlib import Path
import faiss
import numpy as np
from autogen_core import CancellationToken
from llama_cpp import Llama
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from autogen_core.code_executor import CodeBlock
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
# ============================================================
# CONFIG
# ============================================================
BASE = Path("/content")
DB = BASE / "movies.db"
STORE = BASE / "faiss_store"
INDEX_FILE = STORE / "imdb.index"
CHUNKS_FILE = STORE / "chunks.json"
WORK_DIR = BASE / "autogen_work"

KNOWLEDGE = [
    BASE / "imdb_moviesdb_planner_knowledge.md",
    BASE / "imdb_moviesdb_intent_rules.md",
    BASE / "imdb_moviesdb_query_plan_schema.md",
    BASE / "imdb_moviesdb_examples.md",
]

REPO = os.getenv("GGUF_REPO")
GGUF_FILE = os.getenv("GGUF_FILE")
EMBED_MODEL_NAME = os.getenv("EMBED_MODEL_NAME")
TOP_K = 6
MAX_ATTEMPTS = 3

In [6]:
print("[MODEL] loading GGUF model...")
llm_raw = Llama.from_pretrained(
    repo_id=REPO,
    filename=GGUF_FILE,
    n_gpu_layers=-1,
    n_ctx=8192,
    flash_attn=True,
    verbose=False,
)
print("[MODEL] GGUF loaded:", llm_raw.model_path)

[MODEL] loading GGUF model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./gpt-oss-20b-Q8_0.gguf: reconstructing file:   0%|          |  0.00B / 12.1GB            

./gpt-oss-20b-Q8_0.gguf: downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[MODEL] GGUF loaded: /root/.cache/huggingface/hub/models--unsloth--gpt-oss-20b-GGUF/snapshots/d449b42d93e1c2c7bda5312f5c25c8fb91dfa9b4/./gpt-oss-20b-Q8_0.gguf


In [7]:
print("[EMBED] loading embedding model...")
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.embed_model = embed_model
print("[EMBED] loaded:", EMBED_MODEL_NAME)

[EMBED] loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[EMBED] loaded: BAAI/bge-small-en-v1.5


In [72]:
def llm(prompt, json_mode=False, max_tokens=2048):
    kwargs = {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a precise local assistant. "
                    "Do not output reasoning. "
                    "Return only the requested final answer."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.0,
        "top_p": 0.9,
        "top_k": 20,
        "max_tokens": max_tokens,
    }

    if json_mode:
        kwargs["response_format"] = {
            "type": "json_object"
        }

    out = llm_raw.create_chat_completion(**kwargs)

    content = out["choices"][0]["message"]["content"]

    if content is None or not content.strip():
        raise RuntimeError(
            f"GPT-OSS returned empty content.\nFull response:\n{out}"
        )

    content = content.strip()

    marker = "<|channel|>final<|message|>"

    if marker in content:
        content = content.split(marker, 1)[1]

    content = content.replace("<|end|>", "").strip()

    return content

In [74]:
prompt=input('prompt: ')
output=llm(prompt)
print(output)

prompt: hi
Hello! How can I help you today?


In [75]:
def embed(texts):
    # LlamaIndex HuggingFaceEmbedding -> list[list[float]]
    vectors = embed_model.get_text_embedding_batch(texts)
    x = np.asarray(vectors, dtype="float32")
    faiss.normalize_L2(x)
    return x

In [76]:
# ============================================================
# PERSISTENT LOCAL FAISS
# ============================================================
def source_hash():
    h = hashlib.sha256(EMBED_MODEL_NAME.encode())
    for p in KNOWLEDGE:
        if not p.exists():
            raise FileNotFoundError(p)
        h.update(p.read_bytes())
    return h.hexdigest()


def make_chunks():
    chunks = []
    for p in KNOWLEDGE:
        text = p.read_text(encoding="utf-8")
        sections = re.split(r"(?=^#{1,4}\s+)", text, flags=re.M)
        for section in sections:
            section = section.strip()
            if not section:
                continue
            # Long sections are split into ~1800-char blocks.
            for i in range(0, len(section), 1800):
                part = section[i:i + 1800].strip()
                if part:
                    chunks.append({"source": p.name, "text": part})
    return chunks


def load_faiss():
    STORE.mkdir(exist_ok=True)
    current_hash = source_hash()

    if INDEX_FILE.exists() and CHUNKS_FILE.exists():
        meta = json.loads(CHUNKS_FILE.read_text(encoding="utf-8"))
        if meta.get("hash") == current_hash:
            print("[FAISS] local index loaded")
            return faiss.read_index(str(INDEX_FILE)), meta["chunks"]

    print("[FAISS] building local index")
    chunks = make_chunks()
    vectors = embed([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    faiss.write_index(index, str(INDEX_FILE))
    CHUNKS_FILE.write_text(
        json.dumps({"hash": current_hash, "chunks": chunks}, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"[FAISS] saved: {len(chunks)} chunks")
    return index, chunks


def rag(question, index, chunks):
    scores, ids = index.search(embed([question]), min(TOP_K, len(chunks)))
    result = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            c = chunks[int(idx)]
            result.append(
                f"SOURCE={c['source']} SCORE={score:.4f}\n{c['text']}"
            )
    return "\n\n---\n\n".join(result)

In [77]:
def repair_json_text(text):
    text = text.strip()

    # GPT'nin yaptığı basit JSON syntax hatalarını düzelt
    text = re.sub(
        r'("operator"\s*):\s*=\s*(".*?")',
        r'\1: \2',
        text
    )

    return text

In [78]:
def parse_json(text):
    if text is None:
        raise ValueError("LLM response is None")

    text = repair_json_text(text)

    text = text.strip()

    if not text:
        raise ValueError("LLM response is empty")

    # Harmony final channel kaldıysa ayıkla
    marker = "<|channel|>final<|message|>"
    if marker in text:
        text = text.split(marker, 1)[1]

    text = text.replace("<|end|>", "").strip()

    # Markdown ```json ... ``` varsa önce onu al
    fenced = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.I | re.S,
    )

    if fenced:
        candidate = fenced.group(1)

        try:
            obj = json.loads(candidate)

            if isinstance(obj, dict) and "is_related" in obj:
                return obj

        except json.JSONDecodeError:
            pass

    # Direkt JSON olabilir
    try:
        obj = json.loads(text)

        if isinstance(obj, dict) and "is_related" in obj:
            return obj

    except json.JSONDecodeError:
        pass

    # Metindeki TÜM balanced JSON objelerini ara.
    # Sadece planner contract'ına uyanı kabul et.
    required_keys = {
        "is_related",
        "intent",
        "tables",
        "select",
        "joins",
        "filters",
    }

    candidates = []

    for start in range(len(text)):
        if text[start] != "{":
            continue

        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(text)):
            ch = text[i]

            if escape:
                escape = False
                continue

            if ch == "\\" and in_string:
                escape = True
                continue

            if ch == '"':
                in_string = not in_string
                continue

            if not in_string:
                if ch == "{":
                    depth += 1

                elif ch == "}":
                    depth -= 1

                    if depth == 0:
                        candidate = text[start:i + 1]

                        try:
                            obj = json.loads(candidate)

                            if (
                                isinstance(obj, dict)
                                and required_keys.issubset(obj.keys())
                            ):
                                candidates.append(obj)

                        except json.JSONDecodeError:
                            pass

                        break

    if candidates:
        # En büyük / en kapsamlı planner JSON'u seç
        return max(
            candidates,
            key=lambda x: len(json.dumps(x))
        )

    raise ValueError(
        "Could not find valid planner JSON.\n\n"
        f"RAW OUTPUT:\n{text}"
    )

In [79]:
def make_plan(question, context):
    is_direct_sql = bool(
        re.match(
            r"^\s*(SELECT|WITH)\b",
            question,
            flags=re.I
        )
    )

    sql_hint = ""

    if is_direct_sql:
        sql_hint = """
The input is already read-only SQLite SQL.
Analyze it only.
Set intent to "direct_sql_query".
Preserve selected columns, tables, joins,
filters, ordering, aggregation and limit.
"""

    prompt = f"""
You are the planner for Harvard CS50 IMDb movies.db.

Use only the retrieved RAG context.

QUESTION:
{question}

RAG CONTEXT:
{context}

{sql_hint}

Create the smallest correct execution plan.

Return exactly this JSON structure:

{{
  "is_related": true,
  "reason": "",
  "intent": "",
  "tables": [
    {{
      "name": "movies",
      "alias": null
    }}
  ],
  "select": [],
  "joins": [],
  "filters": [],
  "aggregation": null,
  "group_by": [],
  "order_by": [],
  "limit": null,
  "distinct": false,
  "self_join": false,
  "notes": []
}}

Allowed tables:
movies, people, stars, directors, ratings

Rules:
- Use only documented columns and joins.
- Extract all concrete names, titles, years, ratings and limits.
- For two-actor and co-star questions use aliases/self-join.
- "tables" must always contain objects with "name" and optional "alias".
- "select" must contain column names.
- "joins" must contain left/right/type objects.
- "filters" must contain column/operator/value objects.
- "order_by" must contain column/direction objects.
- If no item is required for a list, return [].
- Use null where appropriate.
- Do not generate SQL.
- Do not generate Python.
- Do not output reasoning.
- Do not output markdown.
- Return one valid JSON object only.
"""

    raw = llm(
        prompt,
        json_mode=True,
        max_tokens=2048
    )

    # json.loads(raw) yerine bunu kullan
    plan = parse_json(raw)

    allowed = {
        "movies",
        "people",
        "stars",
        "directors",
        "ratings"
    }

    if plan.get("is_related"):

        table_names = set()

        for table in plan.get("tables", []):

            if isinstance(table, dict):
                name = table.get("name")

            elif isinstance(table, str):
                # fallback: model yine string üretirse sistem çökmesin
                name = table

            else:
                raise ValueError(
                    f"Invalid table definition: {table}"
                )

            if not name:
                raise ValueError(
                    f"Table definition has no name: {table}"
                )

            table_names.add(name)

        unknown_tables = table_names - allowed

        if unknown_tables:
            raise ValueError(
                f"Unknown table(s): {sorted(unknown_tables)}"
            )

        if is_direct_sql:
            plan["intent"] = "direct_sql_query"

    return plan

In [80]:

# ============================================================
# JSON VARIABLE -> PYTHON CODE (SQL IS INSIDE PYTHON)
# ============================================================
def clean_code(text):
    return re.sub(r"^```(?:python)?\s*|\s*```$", "", text.strip(), flags=re.I).strip()


In [81]:

def generate_code(plan):
    return clean_code(llm(f"""
Generate ONE complete Python program from this JSON plan:

{json.dumps(plan, ensure_ascii=False, indent=2)}

Rules:
- Python code only.
- import only os and sqlite3.
- db_path = os.environ["IMDB_DB_PATH"]
- connect READ ONLY with: sqlite3.connect(f"file:{{db_path}}?mode=ro", uri=True)
- SQL must be inside the Python program.
- SELECT / WITH...SELECT only.
- use ? parameters for user values.
- obey tables, joins, filters, distinct, aggregation, ordering, limit and self-join notes.
- print the final result.
- close the connection.
""", max_tokens=2048))

In [82]:

# ============================================================
# SMALL SAFETY CHECK
# ============================================================
FORBIDDEN = [
    "subprocess", "socket", "requests", "urllib", "open(", "os.system", "os.popen",
    " insert ", " update ", " delete ", " drop ", " alter ", " create ",
    " attach ", " detach ", " pragma ", " vacuum ",
]


def check_code(code):
    x = " " + code.lower().replace("\n", " ") + " "
    if "mode=ro" not in x:
        raise ValueError("database connection is not read-only")
    for bad in FORBIDDEN:
        if bad in x:
            raise ValueError(f"forbidden generated code: {bad.strip()}")


In [83]:

# ============================================================
# REPAIR
# ============================================================
def repair(question, context, plan, failed_code, error):
    return clean_code(llm(f"""
Fix the failed Python program below.
The task and JSON plan are unchanged.
SQL must remain INSIDE the Python code.

QUESTION:
{question}

RAG KNOWLEDGE:
{context}

JSON PLAN:
{json.dumps(plan, ensure_ascii=False, indent=2)}

FAILED CODE:
{failed_code}

AUTOGEN ERROR/OUTPUT:
{error}

Return corrected complete Python code only.
Only import os, sqlite3.
Use IMDB_DB_PATH and SQLite mode=ro.
Only SELECT / WITH...SELECT.
Use only documented tables/columns/joins.
""", max_tokens=2048))

In [84]:

# ============================================================
# AUTOGEN EXECUTE / REPAIR LOOP
# ============================================================
async def execute_with_autogen(question, context, plan, code):
    WORK_DIR.mkdir(exist_ok=True)
    os.environ["IMDB_DB_PATH"] = str(DB.resolve())

    executor = LocalCommandLineCodeExecutor(
        work_dir=WORK_DIR, timeout=60, cleanup_temp_files=True
    )
    attempts = []

    try:
        await executor.start()
        for n in range(1, MAX_ATTEMPTS + 1):
            try:
                check_code(code)
            except Exception as e:
                error = f"LOCAL VALIDATION ERROR: {e}"
                attempts.append({"attempt": n, "error": error})
                if n == MAX_ATTEMPTS:
                    raise RuntimeError(error)
                code = repair(question, context, plan, code, error)
                continue

            result = await executor.execute_code_blocks(
                [CodeBlock(language="python", code=code)],
                cancellation_token=CancellationToken(),
            )
            attempts.append({
                "attempt": n,
                "exit_code": result.exit_code,
                "output": result.output,
            })

            if result.exit_code == 0:
                return code, result.output.strip(), attempts

            if n < MAX_ATTEMPTS:
                code = repair(question, context, plan, code, result.output)

        raise RuntimeError(attempts[-1]["output"])
    finally:
        await executor.stop()


In [ ]:
#Show Al Pacino and Robert De Niro birthday

In [85]:

# ============================================================
# FULL FLOW
# ============================================================
async def ask(question):
    # 1) RAG first
    index, chunks = load_faiss()
    context = rag(question, index, chunks)

    # 2) RAG decides relevance + creates JSON variable
    plan = make_plan(question, context)
    print("\n=== JSON PLAN ===")
    print(json.dumps(plan, ensure_ascii=False, indent=2))

    if not plan.get("is_related"):
        print("\nNot related to movies.db:", plan.get("reason"))
        return

    # 3) JSON variable -> Python code containing SQL
    code = generate_code(plan)

    # 4) AutoGen executes Python; error -> LLM repair -> AutoGen executes again
    final_code, output, attempts = await execute_with_autogen(
        question, context, plan, code
    )

    print("\n=== FINAL PYTHON CODE ===")
    print(final_code)
    print("\n=== RESULT ===")
    print(output)
    print("\n=== EXECUTION ATTEMPTS ===")
    print(json.dumps(attempts, ensure_ascii=False, indent=2))


In [86]:

# ============================================================
# COLAB NOTEBOOK HELPER
# ============================================================
async def run_question(question):
    """Colab/Jupyter: await run_question("your question")"""
    return await ask(question)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("question", nargs="?")
    p.add_argument("--rebuild", action="store_true")
    args, _ = p.parse_known_args()

    if args.rebuild:
        INDEX_FILE.unlink(missing_ok=True)
        CHUNKS_FILE.unlink(missing_ok=True)

    if args.question:
        asyncio.run(ask(args.question))
        return

    while True:
        q = input("\nQuestion> ").strip()
        if q.lower() in {"exit", "quit", "q"}:
            break
        if q:
            try:
                asyncio.run(ask(q))
            except Exception as e:
                print("ERROR:", e)

In [87]:
# In Colab/Jupyter use:
#   await run_question("List the names of all people who starred in Toy Story")
#
# When executed as a normal .py script outside Colab, CLI mode is enabled.
if __name__ == "__main__" and "google.colab" not in sys.modules:
    main()

In [88]:
await run_question(
    "List the names of all people who starred in Toy Story"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "Actors are represented through the stars relationship table.",
  "intent": "List the names of all people who starred in Toy Story",
  "tables": [
    {
      "name": "movies"
    },
    {
      "name": "stars"
    },
    {
      "name": "people"
    }
  ],
  "select": [
    "people.name"
  ],
  "joins": [
    {
      "left": "movies.id",
      "right": "stars.movie_id",
      "type": "INNER"
    },
    {
      "left": "stars.person_id",
      "right": "people.id",
      "type": "INNER"
    }
  ],
  "filters": [
    {
      "column": "movies.title",
      "operator": "=",
      "value": "Toy Story"
    }
  ],
  "aggregation": null,
  "group_by": [],
  "order_by": [],
  "limit": null,
  "distinct": false,
  "self_join": false,
  "notes": []
}


/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(



=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
cur = conn.cursor()

sql = """
SELECT people.name
FROM movies
INNER JOIN stars ON movies.id = stars.movie_id
INNER JOIN people ON stars.person_id = people.id
WHERE movies.title = ?
"""

cur.execute(sql, ("Toy Story",))
rows = cur.fetchall()

for row in rows:
    print(row[0])

conn.close()

=== RESULT ===
Tom Hanks
Tim Allen
Don Rickles
Jim Varney

=== EXECUTION ATTEMPTS ===
[
  {
    "attempt": 1,
    "exit_code": 0,
    "output": "Tom Hanks\nTim Allen\nDon Rickles\nJim Varney\n"
  }
]


In [89]:
await run_question(
    "List the titles of all movies in which both Bradley Cooper and Jennifer Lawrence starred"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "Both actors appear in the same movies",
  "intent": "movies_with_two_actors",
  "tables": [
    {
      "name": "movies"
    },
    {
      "name": "stars",
      "alias": "s1"
    },
    {
      "name": "stars",
      "alias": "s2"
    },
    {
      "name": "people",
      "alias": "p1"
    },
    {
      "name": "people",
      "alias": "p2"
    }
  ],
  "select": [
    "movies.title"
  ],
  "joins": [
    {
      "left": "movies.id",
      "right": "s1.movie_id",
      "type": "INNER"
    },
    {
      "left": "movies.id",
      "right": "s2.movie_id",
      "type": "INNER"
    },
    {
      "left": "s1.person_id",
      "right": "p1.id",
      "type": "INNER"
    },
    {
      "left": "s2.person_id",
      "right": "p2.id",
      "type": "INNER"
    }
  ],
  "filters": [
    {
      "column": "p1.name",
      "operator": "=",
      "value": "Bradley Cooper"
    },
    {
      "column": "p2.name",

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(



=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
cursor = conn.cursor()

sql = """
SELECT movies.title
FROM movies
INNER JOIN stars s1 ON movies.id = s1.movie_id
INNER JOIN stars s2 ON movies.id = s2.movie_id
INNER JOIN people p1 ON s1.person_id = p1.id
INNER JOIN people p2 ON s2.person_id = p2.id
WHERE p1.name = ?
  AND p2.name = ?
"""

params = ("Bradley Cooper", "Jennifer Lawrence")
cursor.execute(sql, params)

for row in cursor.fetchall():
    print(row[0])

cursor.close()
conn.close()

=== RESULT ===
Silver Linings Playbook
Serena
American Hustle
Joy

=== EXECUTION ATTEMPTS ===
[
  {
    "attempt": 1,
    "exit_code": 0,
    "output": "Silver Linings Playbook\nSerena\nAmerican Hustle\nJoy\n"
  }
]


In [90]:
await run_question(
    "List the names of all people who starred in a movie in which Kevin Bacon also starred"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "Find all co‑stars of Kevin Bacon.",
  "intent": "List names of all people who starred in a movie in which Kevin Bacon also starred",
  "tables": [
    {
      "name": "people",
      "alias": "target_person"
    },
    {
      "name": "stars",
      "alias": "target_stars"
    },
    {
      "name": "stars",
      "alias": "other_stars"
    },
    {
      "name": "people",
      "alias": "other_people"
    }
  ],
  "select": [
    "other_people.name"
  ],
  "joins": [
    {
      "left": "target_person.id",
      "right": "target_stars.person_id",
      "type": "INNER"
    },
    {
      "left": "target_stars.movie_id",
      "right": "other_stars.movie_id",
      "type": "INNER"
    },
    {
      "left": "other_stars.person_id",
      "right": "other_people.id",
      "type": "INNER"
    }
  ],
  "filters": [
    {
      "column": "target_person.name",
      "operator": "=",
      "value": "Kevin Bacon

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(



=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

query = """
SELECT DISTINCT other_people.name
FROM people AS target_person
INNER JOIN stars AS target_stars
    ON target_person.id = target_stars.person_id
INNER JOIN stars AS other_stars
    ON target_stars.movie_id = other_stars.movie_id
INNER JOIN people AS other_people
    ON other_stars.person_id = other_people.id
WHERE target_person.name = ?
"""

cursor = conn.cursor()
cursor.execute(query, ("Kevin Bacon",))
rows = cursor.fetchall()

for row in rows:
    print(row[0])

conn.close()

=== RESULT ===
Kevin Bacon
Steve Guttenberg
Mickey Rourke
Daniel Stern
Orson Bean
Tommy Citera
Mark Keyloun
David Strathairn
Maria Tucci
Didi Velez
John Lithgow
Lori Singer
Dianne Wiest
Jami Gertz
Rudy Ramos
Paul Rodriguez
Bob Balaban
Michael Beach
Barbara Barrie
Lindsay Crouse
Kyra Sedgwick
Tom Atkins
Sean Astin
K.C. Martel
Jonathan Ward
Alec Baldwin


In [91]:
#SELECT title, year FROM  movies ORDER BY year LIMIT 20
await run_question(
    "SELECT title, year FROM  movies ORDER BY year LIMIT 20"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "The query selects title and year from movies, orders by year ascending, and limits to 20 rows.",
  "intent": "direct_sql_query",
  "tables": [
    {
      "name": "movies",
      "alias": null
    }
  ],
  "select": [
    "title",
    "year"
  ],
  "joins": [],
  "filters": [],
  "aggregation": null,
  "group_by": [],
  "order_by": [
    {
      "column": "year",
      "direction": "ASC"
    }
  ],
  "limit": 20,
  "distinct": false,
  "self_join": false,
  "notes": []
}

=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]

conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
cursor = conn.cursor()

query = """
SELECT title, year
FROM movies
ORDER BY year ASC
LIMIT 20
"""

cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)

cursor.close()
conn.close()

=== RESULT ===
('El huésped del sevillano', 1970)
("Satan's Harvest", 1970)
('The

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(


In [92]:
#SELECT title, year FROM  movies ORDER BY year DESC LIMIT 40
await run_question(
    "SELECT title, year FROM  movies ORDER BY year DESC LIMIT 40"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "Query requests titles and years ordered by year descending with limit 40.",
  "intent": "direct_sql_query",
  "tables": [
    {
      "name": "movies",
      "alias": null
    }
  ],
  "select": [
    "title",
    "year"
  ],
  "joins": [],
  "filters": [],
  "aggregation": null,
  "group_by": [],
  "order_by": [
    {
      "column": "year",
      "direction": "DESC"
    }
  ],
  "limit": 40,
  "distinct": false,
  "self_join": false,
  "notes": []
}

=== FINAL PYTHON CODE ===
import os
import sqlite3

def main():
    db_path = os.environ["IMDB_DB_PATH"]
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        cursor = conn.cursor()
        query = """
            SELECT title, year
            FROM movies
            ORDER BY year DESC
            LIMIT 40
        """
        cursor.execute(query)
        rows = cursor.fetchall()
        for row in rows:
            print(row)
 

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(


In [93]:
#SELECT name, birth FROM people WHERE birth > 0 ORDER BY birth LIMIT 20
await run_question(
    "SELECT name, birth FROM people WHERE birth > 0 ORDER BY birth LIMIT 20"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "",
  "intent": "direct_sql_query",
  "tables": [
    {
      "name": "people",
      "alias": null
    }
  ],
  "select": [
    "name",
    "birth"
  ],
  "joins": [],
  "filters": [
    {
      "column": "birth",
      "operator": ">",
      "value": 0
    }
  ],
  "aggregation": null,
  "group_by": [],
  "order_by": [
    {
      "column": "birth",
      "direction": "ASC"
    }
  ],
  "limit": 20,
  "distinct": false,
  "self_join": false,
  "notes": []
}

=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
cursor = conn.cursor()

query = """
SELECT name, birth
FROM people
WHERE birth > ?
ORDER BY birth ASC
LIMIT 20
"""

cursor.execute(query, (0,))
rows = cursor.fetchall()

for row in rows:
    print(row)

cursor.close()
conn.close()

=== RESULT ===
('Lucio Anneo Seneca', 4)
('Paul Walsh', 21)
('Flavius Jo

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(


In [94]:
#SELECT name, birth FROM  people WHERE name =='Al Pacino' OR name =='Robert De Niro
await run_question(
    "SELECT name, birth FROM  people WHERE name =='Al Pacino' OR name =='Robert De Niro"
)

[FAISS] local index loaded

=== JSON PLAN ===
{
  "is_related": true,
  "reason": "Query requests birth years for two specific people.",
  "intent": "direct_sql_query",
  "tables": [
    {
      "name": "people",
      "alias": null
    }
  ],
  "select": [
    "name",
    "birth"
  ],
  "joins": [],
  "filters": [
    {
      "column": "name",
      "operator": "=",
      "value": "Al Pacino"
    },
    {
      "column": "name",
      "operator": "=",
      "value": "Robert De Niro"
    }
  ],
  "aggregation": null,
  "group_by": [],
  "order_by": [],
  "limit": null,
  "distinct": false,
  "self_join": false,
  "notes": []
}

=== FINAL PYTHON CODE ===
import os
import sqlite3

db_path = os.environ["IMDB_DB_PATH"]
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

query = """
SELECT name, birth
FROM people
WHERE name = ? OR name = ?
"""

params = ("Al Pacino", "Robert De Niro")

cursor = conn.cursor()
cursor.execute(query, params)
rows = cursor.fetchall()

print(rows)

curso

/tmp/ipykernel_10275/961902103.py:8: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(
